# Double Bottom with Breakout Confirmation on SPY
## Strategy Brief
The Double Bottom with Breakout Confirmation strategy aims to identify potential reversal points in the SPY ETF, which tracks the S&P 500 index. The double bottom pattern is a bullish reversal pattern that indicates a potential change in trend from down to up. The strategy involves confirming the pattern with a breakout above the resistance level formed between the two bottoms. Once the breakout is confirmed, the strategy predicts a bullish trend and initiates a long position. The results of this strategy are evaluated against a buy-and-hold approach to assess its effectiveness.
## References
- https://www.merriam-webster.com/dictionary/double

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters and constants necessary for the strategy. This includes the lookback period for identifying the double bottom pattern and the threshold for breakout confirmation.

In [ ]:
LOOKBACK_PERIOD = 20  # Number of days to look back for double bottom pattern
BREAKOUT_THRESHOLD = 0.01  # 1% breakout above resistance level
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'

### PHASE 2 - Data Exploration
We will download historical price data for SPY using the yfinance library. We will then compute the necessary indicators to identify the double bottom pattern and visualize them overlaid on the price chart.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Compute rolling minimum and maximum to identify potential double bottoms
data['Min'] = data['Close'].rolling(window=LOOKBACK_PERIOD).min()
data['Max'] = data['Close'].rolling(window=LOOKBACK_PERIOD).max()

# Plot closing price and indicators
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Close Price')
plt.plot(data['Min'], label='Rolling Min', linestyle='--')
plt.plot(data['Max'], label='Rolling Max', linestyle='--')
plt.title('SPY Price with Double Bottom Indicators')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
In this phase, we develop the logic to generate trading signals based on the double bottom pattern and breakout confirmation. We define the entry and exit logic and create a positions series to track trades.

In [ ]:
def identify_double_bottom(data):
    signals = pd.Series(index=data.index, data=np.nan)
    for i in range(LOOKBACK_PERIOD, len(data)):
        if (data['Close'][i] > data['Max'][i-LOOKBACK_PERIOD:i].max() * (1 + BREAKOUT_THRESHOLD)):
            if (data['Close'][i-LOOKBACK_PERIOD:i].min() == data['Min'][i]):
                signals[i] = 1  # Buy signal
    return signals

# Generate signals
data['Signal'] = identify_double_bottom(data)

# Create positions based on signals
data['Position'] = data['Signal'].ffill().fillna(0)

### PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating the daily returns and plotting the equity curve. The positions are shifted by one day to simulate realistic trading.

In [ ]:
data['Position'] = data['Position'].shift(1)
data['Daily Return'] = data['Close'].pct_change() * data['Position']
data['Equity Curve'] = (1 + data['Daily Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity Curve'], label='Equity Curve')
plt.title('Equity Curve of Double Bottom Strategy')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We evaluate the performance of the strategy using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We compare these metrics against a buy-and-hold strategy.

In [ ]:
def calculate_performance(data):
    cagr = (data['Equity Curve'].iloc[-1] ** (1 / ((data.index[-1] - data.index[0]).days / 365.25))) - 1
    sharpe_ratio = data['Daily Return'].mean() / data['Daily Return'].std() * np.sqrt(252)
    sortino_ratio = data['Daily Return'].mean() / data[data['Daily Return'] < 0]['Daily Return'].std() * np.sqrt(252)
    max_drawdown = (data['Equity Curve'].cummax() - data['Equity Curve']).max() / data['Equity Curve'].cummax().max()
    calmar_ratio = cagr / max_drawdown
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_perf = calculate_performance(data)

# Buy-and-hold strategy
buy_and_hold = data['Close'] / data['Close'].iloc[0]
buy_and_hold_perf = calculate_performance(pd.DataFrame({'Equity Curve': buy_and_hold}))

# Performance comparison
performance_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_perf,
    'Buy and Hold': buy_and_hold_perf
})
print(performance_df)

### PHASE 6 - Deploy & Monitor
We create a function to download the last 60 days of SPY data, compute today's signal, and print the recommended position based on the strategy.

In [ ]:
def get_latest_signal():
    recent_data = yf.download('SPY', period='60d')
    recent_data['Min'] = recent_data['Close'].rolling(window=LOOKBACK_PERIOD).min()
    recent_data['Max'] = recent_data['Close'].rolling(window=LOOKBACK_PERIOD).max()
    recent_data['Signal'] = identify_double_bottom(recent_data)
    recent_data['Position'] = recent_data['Signal'].ffill().fillna(0)
    latest_position = recent_data['Position'].iloc[-1]
    print(f"Today's recommended position: {'Long' if latest_position > 0 else 'Neutral'}")

get_latest_signal()